# 2. Réseaux de neurones convolutifs pour systèmes de recommandations

Ce notebook présente la construction et l'entraînement d'un système de recommandation de films basé sur l'architecture **LightGCN** (Graph Convolutional Networks pour le filtrage collaboratif) en utilisant PyTorch et l'ensemble de données **MovieLens Small**.

### Bases Théoriques de LightGCN :
LightGCN simplifie les GCN classiques en supprimant les transformations de caractéristiques non linéaires et les opérations d'auto-boucles (qui compliquent l'entraînement sans améliorer les performances de filtrage collaboratif). La propagation s'écrit :

$$e_u^{(k+1)} = \sum_{i \in N(u)} \frac{1}{\sqrt{|N(u)|}\sqrt{|N(i)|}} e_i^{(k)}$$
$$e_i^{(k+1)} = \sum_{u \in N(i)} \frac{1}{\sqrt{|N(i)|}\sqrt{|N(u)|}} e_u^{(k)}$$

Où $e_u$ et $e_i$ sont respectivement les embeddings utilisateur et article. Les embeddings finaux sont obtenus par moyenne pondérée des embeddings à chaque couche :

$$e_u = \sum_{k=0}^K \alpha_k e_u^{(k)}; \quad e_i = \sum_{k=0}^K \alpha_k e_i^{(k)}$$

Pour l'apprentissage des préférences par paires, nous minimisons la perte **BPR (Bayesian Personalized Ranking)** :

$$L_{BPR} = -\sum_{u \in U} \sum_{i \in N(u)} \sum_{j \notin N(u)} \ln \sigma(\hat{y}_{ui} - \hat{y}_{uj}) + \lambda \|E^{(0)}\|^2_2$$

In [ ]:
import os
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix

import torch
import torch.nn as nn
import torch.optim as optim

sns_style = plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('ggplot')

## Chargement de MovieLens et Construction du Graphe

In [ ]:
ratings_path = "dataset/ml-latest-small/ratings.csv"
movies_path = "dataset/ml-latest-small/movies.csv"

ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)

print(f"Nombre de notes : {len(ratings)}")
print(f"Nombre de films : {len(movies)}")
ratings.head()

In [ ]:
# Indexation séquentielle des utilisateurs et des films
user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

num_users = len(user_ids)
num_items = len(movie_ids)

user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
idx_to_user = {idx: uid for uid, idx in user_to_idx.items()}

item_to_idx = {iid: idx for idx, iid in enumerate(movie_ids)}
idx_to_item = {idx: iid for iid, idx in item_to_idx.items()}

ratings['user_idx'] = ratings['userId'].map(user_to_idx)
ratings['item_idx'] = ratings['movieId'].map(item_to_idx)

print(f"Utilisateurs uniques : {num_users} | Films uniques : {num_items}")
print(f"Densité du graphe : {(len(ratings)/(num_users*num_items))*100:.3f}%")

In [ ]:
# Création d'un ensemble d'interactions train/test
train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)

# Dictionnaire d'interactions positives
user_train_pos = train_ratings.groupby('user_idx')['item_idx'].apply(list).to_dict()
user_test_pos = test_ratings.groupby('user_idx')['item_idx'].apply(list).to_dict()

## Implémentation de LightGCN dans PyTorch

Nous construisons la matrice d'adjacence normalisée $\tilde{A}$ du graphe bipartite utilisateur-item de taille $(N+M) \times (N+M)$ :

$$A = \begin{pmatrix} 0 & R \\ R^T & 0 \end{pmatrix}$$
$$\tilde{A} = D^{-1/2} A D^{-1/2}$$

In [ ]:
def get_norm_adj_matrix(num_users, num_items, train_df):
    # Matrice d'interaction utilisateur-item R
    users = train_df['user_idx'].values
    items = train_df['item_idx'].values
    
    # Construire la matrice d'adjacence bipartite A
    # L'indexation globale est : [0...num_users-1] pour les utilisateurs, 
    # et [num_users...num_users+num_items-1] pour les articles.
    row = np.concatenate([users, items + num_users])
    col = np.concatenate([items + num_users, users])
    data = np.ones_like(row, dtype=np.float32)
    
    N = num_users + num_items
    adj = coo_matrix((data, (row, col)), shape=(N, N))
    
    # Normalisation symétrique : D^-1/2 * A * D^-1/2
    deg = np.array(adj.sum(axis=1)).flatten()
    deg[deg == 0] = 1.0 # éviter div par 0
    deg_inv_sqrt = np.power(deg, -0.5)
    
    # Matrice diagonale D^-1/2
    deg_inv_sqrt_mat = coo_matrix((deg_inv_sqrt, (np.arange(N), np.arange(N))), shape=(N, N))
    
    norm_adj = deg_inv_sqrt_mat.dot(adj).dot(deg_inv_sqrt_mat)
    
    # Convertir en tenseur creux PyTorch
    coo = norm_adj.tocoo()
    indices = torch.LongTensor([coo.row, coo.col])
    values = torch.FloatTensor(coo.data)
    return torch.sparse_coo_tensor(indices, values, torch.Size([N, N]))

In [ ]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        
        # Initialisation des embeddings à la couche 0
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        
        # Initialisation selon la loi normale
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        
    def forward(self, norm_adj_matrix):
        # Concaténer les embeddings utilisateurs et items de la couche 0
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight
        all_emb = torch.cat([users_emb, items_emb], dim=0)
        
        embs = [all_emb]
        
        # Propagation des embeddings à travers les couches de convolution du graphe
        for layer in range(self.num_layers):
            all_emb = torch.sparse.mm(norm_adj_matrix, all_emb)
            embs.append(all_emb)
            
        # Moyenne finale pondérée (1 / (K + 1))
        embs = torch.stack(embs, dim=1)
        final_embs = torch.mean(embs, dim=1)
        
        # Séparer les embeddings utilisateurs et items finaux
        final_users_emb, final_items_emb = torch.split(final_embs, [self.num_users, self.num_items])
        return final_users_emb, final_items_emb

## Échantillonnage BPR et Entraînement

In [ ]:
# Générateur de paires d'entraînement (utilisateur, article_positif, article_négatif)
def sample_bpr_triplets(num_users, num_items, user_pos_dict, batch_size=2048):
    users = list(user_pos_dict.keys())
    sampled_users = random.choices(users, k=batch_size)
    
    pos_items = []
    neg_items = []
    
    for u in sampled_users:
        pos_item = random.choice(user_pos_dict[u])
        while True:
            neg_item = random.randint(0, num_items - 1)
            if neg_item not in user_pos_dict[u]:
                break
        pos_items.append(pos_item)
        neg_items.append(neg_item)
        
    return torch.LongTensor(sampled_users), torch.LongTensor(pos_items), torch.LongTensor(neg_items)

In [ ]:
# Matrice d'adjacence normalisée pour l'entraînement
norm_adj = get_norm_adj_matrix(num_users, num_items, train_ratings)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LightGCN(num_users, num_items, embedding_dim=64, num_layers=3).to(device)
norm_adj = norm_adj.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.005)
reg_lambda = 1e-4

epochs = 20
batch_size = 2048
loss_history = []

print(f"Entraînement en cours sur {device}...")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    users, pos_items, neg_items = sample_bpr_triplets(num_users, num_items, user_train_pos, batch_size)
    users, pos_items, neg_items = users.to(device), pos_items.to(device), neg_items.to(device)
    
    user_embs, item_embs = model(norm_adj)
    
    u_emb = user_embs[users]
    pos_emb = item_embs[pos_items]
    neg_emb = item_embs[neg_items]
    
    # Prédiction par produit scalaire
    pos_scores = torch.sum(u_emb * pos_emb, dim=1)
    neg_scores = torch.sum(u_emb * neg_emb, dim=1)
    
    # Perte BPR
    bpr_loss = -torch.mean(torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-10))
    
    # Régularisation L2 sur les embeddings originaux (couche 0)
    l2_reg = (model.user_embedding(users).norm(2).pow(2) + 
              model.item_embedding(pos_items).norm(2).pow(2) + 
              model.item_embedding(neg_items).norm(2).pow(2)) / batch_size
    
    loss = bpr_loss + reg_lambda * l2_reg
    
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    if (epoch + 1) % 5 == 0:
        print(f"Époque {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

## Évaluation (Recall@K et NDCG@K)

In [ ]:
def evaluate_recall_ndcg(user_embs, item_embs, test_dict, train_dict, K=20):
    recalls = []
    ndcgs = []
    
    # Scores complets pour tous les utilisateurs-items
    scores = torch.matmul(user_embs, item_embs.t()).cpu().numpy()
    
    for u, actual_items in test_dict.items():
        # Masquer les éléments déjà vus lors de l'entraînement
        seen_items = train_dict.get(u, [])
        u_scores = scores[u].copy()
        u_scores[seen_items] = -np.inf
        
        # Obtenir les K meilleurs éléments prédits
        topk_items = np.argpartition(u_scores, -K)[-K:]
        topk_items = topk_items[np.argsort(u_scores[topk_items])][::-1]
        
        # Calcul du Recall
        hits = len(set(topk_items) & set(actual_items))
        recall = hits / min(len(actual_items), K)
        recalls.append(recall)
        
        # Calcul du NDCG
        dcg = sum([1.0 / np.log2(i + 2) for i, item in enumerate(topk_items) if item in actual_items])
        idcg = sum([1.0 / np.log2(i + 2) for i in range(min(len(actual_items), K))])
        ndcg = dcg / idcg if idcg > 0 else 0.0
        ndcgs.append(ndcg)
        
    return np.mean(recalls), np.mean(ndcgs)

model.eval()
with torch.no_grad():
    final_user_embs, final_item_embs = model(norm_adj)
    
recall, ndcg = evaluate_recall_ndcg(final_user_embs, final_item_embs, user_test_pos, user_train_pos, K=20)
print(f"Recall@20 : {recall:.4f}")
print(f"NDCG@20 : {ndcg:.4f}")

In [ ]:
# Tracer la courbe de perte d'entraînement
plt.figure(figsize=(6, 4))
plt.plot(loss_history, color='blue', label='BPR Loss')
plt.title("Évolution de la Perte pendant l'entraînement")
plt.xlabel("Mise à jour")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Sauvegarde des Embeddings et Cartographie des Films

In [ ]:
os.makedirs("models", exist_ok=True)

assets = {
    "user_embeddings": final_user_embs.cpu().numpy(),
    "item_embeddings": final_item_embs.cpu().numpy(),
    "user_to_idx": user_to_idx,
    "idx_to_user": idx_to_user,
    "item_to_idx": item_to_idx,
    "idx_to_item": idx_to_item,
    "movies_df": movies,
    "ratings_df": ratings,
    "train_interactions": user_train_pos
}

with open("models/recommender_model_assets.pkl", "wb") as f:
    pickle.dump(assets, f)
print("Tous les actifs de recommandation GCN sauvegardés dans models/recommender_model_assets.pkl")